<a href="https://colab.research.google.com/github/spokenprim618/python100-colabHW/blob/main/CTD_Assignment_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
johnmcgarvey55_code_the_dream_assignment_6_path = kagglehub.dataset_download('johnmcgarvey55/code-the-dream-assignment-6')

print('Data source import complete.')


Using Colab cache for faster access to the 'code-the-dream-assignment-6' dataset.
Data source import complete.


In [5]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
try:
    from thefuzz import process
except ImportError:
    !pip install thefuzz
    from thefuzz import process
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/code-the-dream-assignment-6/name_and_address_3.csv
/kaggle/input/code-the-dream-assignment-6/name_and_address_0.csv
/kaggle/input/code-the-dream-assignment-6/name_and_address_2.csv
/kaggle/input/code-the-dream-assignment-6/name_and_address_1.csv


## Task 1

The first idea is to compare the 4 csv to make sure they are all holding the same information just different errors. If they are not all the same I will have to find where the common entries are to see if I can use different parts of the csv entries to try and sticth up the info.

I can go through each csv to remove the incomplete or duplicate data

I can give a default to the wrong information

I can check and merge information from different files

I can just use the most complete csv

## Task 2

In [6]:
df0 = pd.read_csv('/root/.cache/kagglehub/datasets/johnmcgarvey55/code-the-dream-assignment-6/versions/1/name_and_address_0.csv')
df1 = pd.read_csv('/root/.cache/kagglehub/datasets/johnmcgarvey55/code-the-dream-assignment-6/versions/1/name_and_address_1.csv')
df2 = pd.read_csv('/root/.cache/kagglehub/datasets/johnmcgarvey55/code-the-dream-assignment-6/versions/1/name_and_address_2.csv')
df3 = pd.read_csv('/root/.cache/kagglehub/datasets/johnmcgarvey55/code-the-dream-assignment-6/versions/1/name_and_address_3.csv')
print(df0.head(5))
print('________________________________')
print(df1.head(5))
print('________________________________')

print(df2.head(5))
print('________________________________')

print(df3.head(5))
print('________________________________')

df = pd.concat([df0,df1,df2,df3])
df_save = df.copy()
print(df.info())

                Name                                     Address      Zip  \
0     Jerry Bartlett         009 Tristan Meadow Martinezbury, NV  97263.0   
1  Jennifer Gonzalez  0098 Finley Valleys Port Michelleshire, ID  97726.0   
2          Jack Reed  0098 Finley Valleys Port Michelleshire, ID  97726.0   
3    Mrs. Gail Perez          03284 Carson Fields Acostaview, AK      NaN   
4      Andrew Martin          03284 Carson Fields Acostaview, AK  45268.0   

                   Phone  
0    (616)477-6740x83474  
1             6094074870  
2           432.845.9420  
3  +1-826-533-8606x22246  
4                    NaN  
________________________________
                Name                                       Address      Zip  \
0     Jerry Bartlett         009 TristXXan Meadow Martinezbury, NV  97263.0   
1  Jennifer Gonzalez    0098 Finley Valleys Port Michelleshire, ID      NaN   
2          Jack Reed  0098 FinleXXy Valleys Port Michelleshire, ID  97726.0   
3  Mrs. GXXail Perez      

## Task 3

In [7]:
df_names = df.value_counts('Name')
names = list(df_names[df_names > 2].index)
print(names[0:10])
df['Name'] = df['Name'].map(lambda x : x if x in names else process.extractOne(x, names)[0])
df_addresses = df.value_counts('Address')
address = list(df_addresses[df_addresses >2].index)
df['Address'] = df['Address'].map(lambda x: x if x in address else process.extractOne(x, address)[0])


['Charles Smith', 'Rebecca Carey', 'Pamela Costa', 'Jennifer Levy', 'Brandi Sanders', 'Thomas Jones', 'April Lee', 'Jacob Hardy', 'Samuel Brown', 'Alan Carpenter']


## Task 4

In [8]:
def fix_anomaly(group):
    group_na = group.dropna()
    if group_na.empty:
        return group.values
    mode = group_na.mode()
    if mode.empty:
        return group.values
    return mode.iloc[0]

df['Zip'] = df.groupby(['Name', 'Address'], as_index=False)['Zip'].transform(fix_anomaly).reset_index(drop=True)
df['Phone'] = df.groupby(['Name', 'Address'], as_index=False)['Phone'].transform(fix_anomaly).reset_index(drop=True)
print(df['Zip'].head(10))
print(df['Phone'].head(10))


0    97263.0
1    97726.0
2    97726.0
3    45268.0
4    45268.0
5    45268.0
6    98744.0
7    98744.0
8    98744.0
9    82126.0
Name: Zip, dtype: float64
0      (616)477-6740x83474
1               6094074870
2             432.845.9420
3    +1-826-533-8606x22246
4             705-208-6392
5        298-984-3837x5183
6          +1-562-620-5663
7          +1-371-732-8951
8               8706775285
9        686-401-2951x1016
Name: Phone, dtype: object


## Task 5

In [9]:
df = df.drop_duplicates().reset_index(drop=True)

print(df.head(10))
print(df.info())


                      Name                                       Address  \
0           Jerry Bartlett           009 Tristan Meadow Martinezbury, NV   
1        Jennifer Gonzalez    0098 Finley Valleys Port Michelleshire, ID   
2                Jack Reed    0098 Finley Valleys Port Michelleshire, ID   
3  Mrs. Taylor Johnson DDS            03284 Carson Fields Acostaview, AK   
4            Andrew Martin            03284 Carson Fields Acostaview, AK   
5             Linda Barton            03284 Carson Fields Acostaview, AK   
6         Mrs. Tammy Davis  048 David Burgs Suite 890 Lake Nancystad, DC   
7               Devin Ruiz  048 David Burgs Suite 890 Lake Nancystad, DC   
8               Jacob Todd  048 David Burgs Suite 890 Lake Nancystad, DC   
9            Kevin Andrews               0528 Snow Place Michaeltown, NH   

       Zip                  Phone  
0  97263.0    (616)477-6740x83474  
1  97726.0             6094074870  
2  97726.0           432.845.9420  
3  45268.0  +1-826-

## Task 6

In [10]:
rows_with_nulls = df[df.isnull().any(axis=1)]
print(rows_with_nulls)
print(df[df['Name']=='Tammie ThoXXmas'])
print(df[df['Name']=='Charles Smith'])


Empty DataFrame
Columns: [Name, Address, Zip, Phone]
Index: []
                Name                                     Address      Zip  \
403  Tammie ThoXXmas      15144 Alexandria Glens South Jason, MN  57094.0   
405  Tammie ThoXXmas  45165 Carly Lodge Apt. 451 Kennethberg, MA  57575.0   

                  Phone  
403        922.910.9364  
405  495-362-0481x98295  
              Name                                          Address      Zip  \
12   Charles Smith       05678 Mccullough Branch Elizabethshire, VA   6636.0   
37   Charles Smith     1144 Martin Villages Apt. 102 Hebertport, NM  22162.0   
93   Charles Smith           25724 Mary Flat Apt. 920 Jamestown, GA  13131.0   
216  Charles Smith       612 Walter Grove Suite 736 Morrismouth, RI  74843.0   
323  Charles Smith                   323 Aaron Tunnel West Alex, GA  42138.0   
377  Charles Smith                              USNV Johnson FPO AP  44100.0   
412  Charles Smith  140 Larry Center Suite 787 West Charlesbury, MD

## Task 7

In the case of tammie we grouped the entries by both name and address so these two are actually seperated by their address. They could be different or the same person or a change of address for the same person. Charles is the same idea all have different addresses.

There could be a generalization of removing address so all names are grouped the same and are changed according to the mode or no change at all that these are different people but make sure that the area is consistent so if they do live in an area that the phone and zip are in accordance.

IN the case of "Mrs. Gail Perez to  "Mrs. Taylor Johnson DDS" I think this change happened because of the way process.extractOne(x, address)[0] functions. So its based on the percentage of a match but we don't have anything to validate so any percentage of match no matter how low will create a change and it just so happens both names have Mrs. and "Mrs. Taylor Johnson DDS" was the dominant name and the misspells were twice not 3.
To avoid this we needed a threshold to make sure a certain percentage of match is allowed to go through with a change.

I don't know what this one check could be but I do see that the assumptions of the thresholds and the lack of thresholds caused errors.

## Task 8

In [13]:
log_entries = pd.Series([
    "[2023-10-26 10:00:00] INFO: User logged in",
    "[2023-10-26 10:05:30] WARNING: Invalid input",
    "[2023-10-26 10:10:15] ERROR: Database connection failed",
    "[2023-10-26 10:12:45] DEBUG: Processing request"
])

regex_logs = r"\[(?P<timestamp>.*?)\]\s(?P<level>\w+):\s(?P<message>.*)"
extracted_logs = log_entries.str.extract(regex_logs, expand=True)
print("Extracted logs:")
print(extracted_logs)
print('________________________________')


text_data = pd.Series([
    "Value is {amount}.",
    "The price is [value].",
    "Cost: (number)",
    "Quantity = <qty>"
])
placeholder_pattern = r"[\{\[\(<].*?[\}\]\)>]"
standardized_text = text_data.str.replace(placeholder_pattern, "<VALUE>", regex=True)
print("Standardized text:")
print(standardized_text)
print('________________________________')

df = pd.DataFrame({
    "order_id": [123, 124, 125, 126],
    "customer_name": ["Alice", "Bob", "Charlie", "Diana"],
    "order_status": ["shipped", "cancelled", "shipped", "delivered"],
    "created_at": ["2021-01-05", "2021-01-06", "2021-01-06", "2021-01-07"],
    "updated_at": ["2021-01-07", "2021-01-07", "2021-01-08", "2021-01-08"]
})

time_columns = df.filter(regex=r"_at$")
print("Columns ending with _at:")
print(time_columns)
print('________________________________')


order_data = [
    "Order #123 has been shipped on 2021-01-05 (Tuesday)",
    "Order #124 has been cancelled",
    "shipment confirmation #125 on 02/06/2021",
    "Order #126 delivered on 01 07 2021",
    "Canceled order #127, refund pending",
    "order #128 - Shipped 2021/03/10"
]

orders = pd.Series(order_data)
regex_orders = r"(?:Order|order|shipment confirmation)\s*#(?P<order_number>\d+).*?(?P<date>\d{2,4}[-/ ]\d{2}[-/ ]\d{2,4}).*?(?P<shipped>shipped|delivered)?"
order_table = orders.str.extract(regex_orders, expand=True)

order_table['order_number'] = order_table['order_number'].astype(float).astype('Int64')  # nullable int
order_table['shipped'] = order_table['shipped'].notnull()
order_table['date'] = pd.to_datetime(order_table['date'], errors='coerce', dayfirst=False)
order_table = order_table.dropna(subset=['date'])

print("Order table:")
print(order_table)

shipped_orders = orders[orders.str.contains("shipped", case=False)]
print("Shipped orders:")
print(shipped_orders)
print('________________________________')


Extracted logs:
             timestamp    level                     message
0  2023-10-26 10:00:00     INFO              User logged in
1  2023-10-26 10:05:30  WARNING               Invalid input
2  2023-10-26 10:10:15    ERROR  Database connection failed
3  2023-10-26 10:12:45    DEBUG          Processing request
________________________________
Standardized text:
0        Value is <VALUE>.
1    The price is <VALUE>.
2            Cost: <VALUE>
3       Quantity = <VALUE>
dtype: object
________________________________
Columns ending with _at:
   created_at  updated_at
0  2021-01-05  2021-01-07
1  2021-01-06  2021-01-07
2  2021-01-06  2021-01-08
3  2021-01-07  2021-01-08
________________________________
Order table:
   order_number       date  shipped
0           123 2021-01-05    False
Shipped orders:
0    Order #123 has been shipped on 2021-01-05 (Tue...
5                      order #128 - Shipped 2021/03/10
dtype: object
________________________________
